In [3]:
import numpy as np
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
from skimage.transform import resize

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.models import Model
from keras.applications import ResNet50
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adamax
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [ ]:
DTRAIN_PATH = ('../data/Training/')
DTEST_PATH = ('../data/Testing/')

X_train = []
Y_train = []

img_rows = 224
img_cols = 224
classes = os.listdir(DTRAIN_PATH)


print(classes)

['notumor', 'meningioma', 'pituitary', 'glioma']


In [ ]:
# LOAD DATA

# Load Data Train
for class_name in classes:
    class_path = os.path.join(DTRAIN_PATH, class_name)
    class_num = classes.index(class_name)

    for img_path in os.listdir(class_path):
      img_full_path = plt.imread(os.path.join(class_path, img_path))
      img_resized = resize(img_full_path, (img_rows, img_cols, 3))

      X_train.append(img_resized)
      Y_train.append(class_num)

# Load Data Test

for class_name in classes:
    class_path = os.path.join(DTEST_PATH, class_name)
    class_num = classes.index(class_name)

    for img_path in os.listdir(class_path):
      img_full_path = plt.imread(os.path.join(class_path, img_path))
      img_resized = resize(img_full_path, (img_rows, img_cols, 3))

      X_train.append(img_resized)
      Y_train.append(class_num)


X_train_train = np.array(X_train)
Y_train = np.array(Y_train)


X_train.shape

In [ ]:
# Shuffling Data

X_train, Y_train = shuffle(X_train, Y_train, random_state=42)

# Check Data
X_train.shape

In [ ]:
# Plotting Image
# Untuk mengecek apakah sudah sesuai

plt.figure(figsize=(10, 10))

for i in range(15):
  plt.subplot(5, 3, i+1)
  plt.imshow(X_train[i])
  plt.title(classes[Y_train[i]])
  plt.axis('off')

plt.show()

In [ ]:
# Splitting Data

X_train, X_temp, Y_train, Y_test = train_test_split(x_train, y_train, test_size=0.2, random_state=42)

X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.1, random_state=42)

In [ ]:
# Data Augmentation

data_Augment = ImageDataGenerator(
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

data_Augment.fit(X_train)

# Ubah ke Categorical

y_train_new = to_categorical(Y_train, num_classes=4)
y_val_new = to_categorical(Y_val, num_classes=4)
y_test_new = to_categorical(Y_test, num_classes=4)


In [ ]:
# Buat Model

base_model = ResNet50(input_shape=(img_rows, img_cols, 3), include_top=False, weights='imagenet')

for layer in base_model.layers[-30:]:
  layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')
])

model.compile(Adamax(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint('Bmodel.h5', monitor='val_loss', save_best_only=True)

model.summary()


In [ ]:
# Training the Model

history = model.fit(data_Augment.flow(X_train, Y_train, batch_size=32), epochs=50, validation_data=(X_val, Y_val), callbacks=[early_stopping, checkpoint])

In [ ]:
# Save the Model

model = load_model('Bmodel.h5')

In [ ]:
# Visualization

model.evaluate(X_test, Y_test_new)

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Confusion Matrix

y_pred =  model.predit(X_test)
y_pred = np.argmax(y_pred, axis=1)

conf_m = confusion_matrix(Y_test, y_pred)
sns.heatmap(conf_m, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Classification Report

print(classification_report(Y_test, y_pred, target_names=labels))

conf_m = confusion_matrix(Y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_m, display_labels=classes)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix")
plt.show()

print(history_ft.history["val_accuracy"][-3:])
print(history_ft.history["val_loss"][-3:])

In [ ]:
# Visualization Model Predictions

plt.figure(figsize=(15, 15))

for i in range(20):
  plt.subplot(5, 4, i+1)
  plt.imshow(X_test[i])
  plt.title(f"Truth: {classes[Y_test[i]]}\nPredicted: {classes[y_pred[i]]}")
  plt.axis('off')
  plt.show()